In [ ]:
%load_ext autoreload
%autoreload 2

# Readme

Compare exported and SQL tables

In [1]:
from IPython.display import clear_output
from djimaging.user.alpha.utils import database as sql_database

# Choose indicator by uncommenting one of the following lines
indicator = 'calcium'
# indicator = 'glutamate'

sql_database.connect_dj(indicator=indicator)
clear_output()

In [2]:
from djimaging.user.alpha.utils.database_no_sql import DatabaseInterface
no_sql_database = DatabaseInterface(indicator)

Loading calcium data from: /gpfs01/berens/user/joesterle/projects/alpha/__public_s-on-alpha-regional/gnode/s-on-alpha-regional/database-exports/Ran/sONa_calcium


In [6]:
import inspect
import itertools

def get_target_functions(obj):
    """
    Extract all functions from a class instance that start with 'get_' and end with '_tab'
    """
    target_functions = {}

    for name in dir(obj):
        if name.startswith('get_') and name.endswith('_tab'):
            func = getattr(obj, name)
            if callable(func):
                target_functions[name] = func

    return target_functions

def get_function_params(func):
    """
    Extract parameters from a function, excluding 'self' if it exists
    """
    signature = inspect.signature(func)
    parameters = {}

    for param_name, param in signature.parameters.items():
        if param_name != 'self':
            # If parameter has a default value and is a boolean, test both values
            if param.default is not inspect.Parameter.empty:
                if isinstance(param.default, bool) or param_name.endswith('_filter'):
                    parameters[param_name] = [True, False]
                else:
                    parameters[param_name] = [param.default]
            # For boolean parameters without defaults, test both True and False
            elif param.annotation == bool or param_name.endswith('_filter'):
                parameters[param_name] = [True, False]
            # For other parameters without defaults, we'll use None as a placeholder
            else:
                parameters[param_name] = [None]

    return parameters

def generate_parameter_combinations(params_dict):
    """
    Generate all possible combinations of parameters
    """
    param_names = list(params_dict.keys())
    param_values = [params_dict[name] for name in param_names]

    for values_combination in itertools.product(*param_values):
        yield {name: value for name, value in zip(param_names, values_combination)}

def run_comparison_tests():
    """
    Test all target functions with all parameter combinations
    """
    # Get all target functions from no_sql_database instance
    no_sql_functions = get_target_functions(no_sql_database)

    print(f"Found {len(no_sql_functions)} target functions in no_sql_database")

    successful_tests = 0
    failed_tests = 0

    for func_name, no_sql_func in no_sql_functions.items():
        print(f"\nTesting function: {func_name}")

        # Check if the same function exists in sql_database
        if not hasattr(sql_database, func_name):
            print(f"  Function {func_name} does not exist in sql_database. Skipping.")
            continue

        sql_func = getattr(sql_database, func_name)

        # Get parameters for the no_sql function
        params = get_function_params(no_sql_func)
        print(f"  Parameters found: {list(params.keys())}")

        # Generate all parameter combinations
        param_combinations = list(generate_parameter_combinations(params))
        print(f"  Testing {len(param_combinations)} parameter combinations")

        # Test each parameter combination
        for i, param_combo in enumerate(param_combinations, 1):
            param_str = ", ".join(f"{k}={v}" for k, v in param_combo.items())
            print(f"  Combination {i}/{len(param_combinations)}: {param_str}")

            try:
                # Call both functions with the same parameters
                no_sql_results = no_sql_func(**param_combo)
                sql_results = sql_func(**param_combo)

                if not isinstance(no_sql_results, tuple):
                    no_sql_results = (no_sql_results, )
                    sql_results = (sql_results, )

                for no_sql_result, sql_result in zip(no_sql_results, sql_results):
                    # Compare the lengths
                    assert len(no_sql_result) == len(sql_result), \
                        f"Length mismatch: no_sql={len(no_sql_result)}, sql={len(sql_result)}"

                    print(f"    ✓ Test passed: Both returned {len(no_sql_result)} items")
                    successful_tests += 1

            except Exception as e:
                print(f"    ✗ Test failed: {str(e)}")
                failed_tests += 1

    # Print summary
    total_tests = successful_tests + failed_tests
    print(f"\nTest Summary:")
    print(f"  Total tests: {total_tests}")
    print(f"  Successful: {successful_tests} ({successful_tests/total_tests*100:.1f}%)")
    print(f"  Failed: {failed_tests} ({failed_tests/total_tests*100:.1f}%)")

In [7]:
run_comparison_tests()

Found 21 target functions in no_sql_database

Testing function: get_averages_tab
  Parameters found: ['quality_filter']
  Testing 2 parameter combinations
  Combination 1/2: quality_filter=True
    ✓ Test passed: Both returned 1526 items
  Combination 2/2: quality_filter=False
    ✓ Test passed: Both returned 1543 items

Testing function: get_clustering_features_tab
  Parameters found: []
  Testing 1 parameter combinations
  Combination 1/1: 
    ✓ Test passed: Both returned 1 items

Testing function: get_clustering_params_tab
  Parameters found: []
  Testing 1 parameter combinations
  Combination 1/1: 
    ✓ Test passed: Both returned 1 items

Testing function: get_clustering_tab
  Parameters found: []
  Testing 1 parameter combinations
  Combination 1/1: 
    ✓ Test passed: Both returned 645 items

Testing function: get_experiment_tab
  Parameters found: []
  Testing 1 parameter combinations
  Combination 1/1: 
    ✓ Test passed: Both returned 12 items

Testing function: get_field_rf